In [1]:
import csv
import os
import random

# ── Config ────────────────────────────────────────────────────────────────────
DATE        = 20240207
SYMBOL      = "SH.510050"
TICK_SIZE   = 0.001          # ETF tick size
BASE_PRICE  = 2.950          # starting mid price
LEVELS      = 10

ORDERS_PER_WINDOW    = (3, 8)   # random order count per 3-sec window
MAKER_RATIO          = 0.7      # 70% maker, 30% taker
STOP_IN_MAKER_RATIO  = 0.1      # 10% of makers are stop orders

from pathlib import Path

# 获取 Notebook 所在的当前目录，并在里面创建 sample_data
OUTPUT_DIR = Path.cwd() / "sample_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Helpers ───────────────────────────────────────────────────────────────────
def round_tick(price):
    return round(round(price / TICK_SIZE) * TICK_SIZE, 6)

def to_ms(ts):
    ms = ts % 1000
    s  = (ts // 1000) % 100
    m  = (ts // 100000) % 100
    h  = ts // 10000000
    return h * 3600000 + m * 60000 + s * 1000 + ms

def from_ms(t):
    h = t // 3600000; t %= 3600000
    m = t // 60000;   t %= 60000
    s = t // 1000;    ms = t % 1000
    return h * 10000000 + m * 100000 + s * 1000 + ms

def ts_to_str(ts):
    ms = ts % 1000
    s  = (ts // 1000) % 100
    m  = (ts // 100000) % 100
    h  = ts // 10000000
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"

def random_time_between(ts_cur, ts_next):
    lo, hi = to_ms(ts_cur) + 1, to_ms(ts_next) - 1
    return from_ms(random.randint(lo, hi)) if lo <= hi else ts_cur

def generate_time_slots():
    """All 3-second timestamps for both A-share trading sessions."""
    slots = []
    for sh, sm, eh, em in [(9, 30, 11, 30), (13, 0, 15, 0)]:
        t   = sh * 3600 + sm * 60
        end = eh * 3600 + em * 60
        while t <= end:
            h, m, s = t // 3600, (t % 3600) // 60, t % 60
            slots.append(h * 10000000 + m * 100000 + s * 1000)
            t += 3
    return slots

def build_order_book(mid):
    """Return (bids_price, bids_vol, asks_price, asks_vol) for 10 levels."""
    bp1 = round_tick(mid - TICK_SIZE)
    ap1 = round_tick(bp1 + TICK_SIZE)
    bids_p = [round_tick(bp1 - i * TICK_SIZE) for i in range(LEVELS)]
    asks_p = [round_tick(ap1 + i * TICK_SIZE) for i in range(LEVELS)]
    bids_v = [max(100, int(random.lognormvariate(8, 1)) // 100 * 100) for _ in range(LEVELS)]
    asks_v = [max(100, int(random.lognormvariate(8, 1)) // 100 * 100) for _ in range(LEVELS)]
    return bids_p, bids_v, asks_p, asks_v

# ── Level2 headers ────────────────────────────────────────────────────────────
L2_HEADERS = (
    ["date", "symbol", "time", "last", "volume", "amount"]
    + [f"bp{i}" for i in range(1, 11)]
    + [f"bv{i}" for i in range(1, 11)]
    + [f"ap{i}" for i in range(1, 11)]
    + [f"av{i}" for i in range(1, 11)]
)
ORDER_HEADERS = ["date", "symbol", "time_str", "side", "price", "quantity", "order_type"]
TRADE_HEADERS = ["date", "symbol", "time_str", "price", "quantity"]

# ── Main ──────────────────────────────────────────────────────────────────────
def generate_all():
    slots      = generate_time_slots()
    l2_rows    = []
    order_rows = []
    trade_rows = []

    mid        = BASE_PRICE
    cum_vol    = 0
    cum_amount = 0.0
    last_price = BASE_PRICE

    for i, ts in enumerate(slots):
        # ── 1. Level2 snapshot ──────────────────────────────────────────────
        mid = round_tick(mid + random.gauss(0, mid * 0.0003))
        mid = max(mid, 0.1)

        bids_p, bids_v, asks_p, asks_v = build_order_book(mid)
        bp1, ap1 = bids_p[0], asks_p[0]

        trade_vol   = max(0, int(random.lognormvariate(6, 1)) // 100 * 100)
        last_price  = round_tick(random.uniform(bp1, ap1))
        cum_vol    += trade_vol
        cum_amount += trade_vol * last_price

        row = {
            "date": DATE, "symbol": SYMBOL, "time": ts,
            "last": round(last_price, 6),
            "volume": cum_vol,
            "amount": round(cum_amount, 2),
        }
        for j in range(LEVELS):
            row[f"bp{j+1}"] = round(bids_p[j], 6)
            row[f"bv{j+1}"] = bids_v[j]
            row[f"ap{j+1}"] = round(asks_p[j], 6)
            row[f"av{j+1}"] = asks_v[j]
        l2_rows.append(row)

        # ── 2. Orders & Trades within this 3-sec window ─────────────────────
        ts_next = slots[i + 1] if i + 1 < len(slots) else ts + 3000

        for _ in range(random.randint(*ORDERS_PER_WINDOW)):
            side     = random.choice(["buy", "sell"])
            is_taker = random.random() > MAKER_RATIO
            ot       = random_time_between(ts, ts_next)
            tstr     = ts_to_str(ot)
            qty      = int(random.choice([100, 200, 300, 500, 1000]) * random.randint(1, 5))

            if is_taker:
                # Taker: willing to cross the spread → market order
                if side == "buy":
                    price = round_tick(ap1 + random.randint(0, 2) * TICK_SIZE)
                else:
                    price = round_tick(bp1 - random.randint(0, 2) * TICK_SIZE)
                otype = "market"

                # Matching trade
                trade_price = ap1 if side == "buy" else bp1
                trade_qty   = qty if random.random() < 0.7 else max(100, int(qty * random.uniform(0.3, 0.9) // 100 * 100))
                trade_rows.append({
                    "date": DATE, "symbol": SYMBOL,
                    "time_str": tstr,
                    "price": round(trade_price, 6),
                    "quantity": trade_qty,
                })
            else:
                # Maker: patient, queues inside or beyond the spread → limit / stop
                is_stop = random.random() < STOP_IN_MAKER_RATIO
                offset  = random.randint(6, 15) if is_stop else random.randint(1, 5)
                price   = round_tick(bp1 - offset * TICK_SIZE) if side == "buy" else round_tick(ap1 + offset * TICK_SIZE)
                otype   = "stop" if is_stop else "limit"

            order_rows.append({
                "date": DATE, "symbol": SYMBOL,
                "time_str": tstr, "side": side,
                "price": round(price, 6),
                "quantity": qty, "order_type": otype,
            })

    # ── Write CSVs ────────────────────────────────────────────────────────────
    def write_csv(path, headers, rows):
        with open(path, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=headers)
            w.writeheader()
            w.writerows(rows)
        print(f"✓ {os.path.basename(path):<12} {len(rows):>7,} rows  →  {path}")

    write_csv(os.path.join(OUTPUT_DIR, "level2.csv"), L2_HEADERS, l2_rows)
    write_csv(os.path.join(OUTPUT_DIR, "order.csv"),  ORDER_HEADERS, order_rows)
    write_csv(os.path.join(OUTPUT_DIR, "trade.csv"),  TRADE_HEADERS, trade_rows)

if __name__ == "__main__":
    generate_all()

✓ level2.csv     4,802 rows  →  /Users/chongjidelaoshu/Desktop/my-first-project/scripts/sample_data/level2.csv
✓ order.csv     26,160 rows  →  /Users/chongjidelaoshu/Desktop/my-first-project/scripts/sample_data/order.csv
✓ trade.csv      7,906 rows  →  /Users/chongjidelaoshu/Desktop/my-first-project/scripts/sample_data/trade.csv
